# FIRE-3D BEAM — 공간(SPA: 실·계단실·승강기) 분할 모델 GPU 학습 (Google Colab)

**먼저: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후 저장.** 그다음 위에서부터 셀을 차례로 실행(▶)합니다. 셀마다 진행 상황을 출력합니다.

- 준비물(Google Drive `내 드라이브/FIRE-3D-BEAM/`)
  - `FIRE-3D-BEAM-dev/` 저장소 폴더(이번에 고친 `evalr/spa_train.py·spa_data.py·spa_eval.py·spa_colab.py·spa_held198.json`, `fire3dbeam/spa_seg.py` 포함, `models/` 포함) 또는 `FIRE-3D-BEAM-dev_github.zip`
  - `239.건축 도면 데이터/01-1.정식개방데이터/Training/01.원천데이터/TS_SPA_1.zip·TS_SPA_2.zip`, `Training/02.라벨링데이터/TL_SPA.zip`, `Validation/02.라벨링데이터/VL_SPA.zip`
- 결과: `FIRE-3D-BEAM/colab_results/spa_gpu/` — `ckpt/last.pt·best.pt`(5분마다 저장), `spa_seg_gpu.onnx`+`.json`, `score_held198.json`, `train_log.txt`, `cache/*.tar`(추출 캐시)
- **연결이 끊기면**: 1)~3) 셀을 다시 실행 → 4)는 Drive 의 `cache/*.tar` 에서 바로 복원(zip 을 다시 읽지 않음) → 5)는 `last.pt` 에서 자동으로 이어 학습(시간 예산 누적).
- 보류(held) 영상은 파일명 md5 분할로 학습에서 빠지고, 이전 보고(v2)의 **보류 198장**(`evalr/spa_held198.json`)은 평가에만 씁니다.
- 영상 축소는 로컬 축소본과 같은 방법(PIL LANCZOS, 긴 변 2048) → 7)에서 기존 v2 모델을 다시 채점하면 로컬 보고 수치(.748 등)와 같아야 정상.
- 예상 시간(T4): 1~3) 5분 · 4) 추출 30~45분(다음부터 tar 복원 1~2분) · 5) 학습 120분 · 6)~7) 약 10분 → **처음 약 2시간 50분**.
- ※ AI허브 자료는 이 Colab(본인 계정) 안에서만 사용 — 외부 공개·재배포 금지. 실제 소방활동 정보카드는 올리지 마세요.

In [ ]:
# 1) 설정 + Drive 연결 + GPU 확인
import os, sys, json, time, shutil, subprocess, zipfile, glob
try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False
BASE = os.environ.get('FIRE3DBEAM_BASE', '/content/drive/MyDrive/FIRE-3D-BEAM')     # 로컬 시험: FIRE3DBEAM_BASE=가짜 Drive 폴더
WORK = os.environ.get('FIRE3DBEAM_WORK', '/content')                              # 작업(임시) 폴더 — Colab VM 디스크
if IN_COLAB and 'FIRE3DBEAM_BASE' not in os.environ:
    from google.colab import drive; drive.mount('/content/drive')
DATA = BASE + '/239.건축 도면 데이터/01-1.정식개방데이터'
OUTD = BASE + '/colab_results/spa_gpu'; os.makedirs(OUTD + '/ckpt', exist_ok=True); os.makedirs(OUTD + '/cache', exist_ok=True)
CODE = WORK + '/fire3dbeam'; AI = WORK + '/aihub'; IMG = AI + '/img/SPA'; CACHE = WORK + '/cache/spa_seg'
E = lambda k, d: type(d)(os.environ.get(k, d))
# ── 학습 설정 (T4 16GB · 약 2시간). 로컬 시험은 환경변수로 덮어씀 ──
MINUTES   = E('SPA_MINUTES', 120.0)   # 학습 시간 예산(분, 끊겨도 누적)
DEVICE    = E('SPA_DEVICE', 'cuda')
WIDTH     = E('SPA_WIDTH', 2.0)       # 채널 배율: 1.0 = 기존 0.54M, 2.0 ≈ 2.15M 파라미터
BS        = E('SPA_BS', 32)
LR        = E('SPA_LR', 3e-3)
WORKERS   = E('SPA_WORKERS', 2)       # Colab vCPU 2개
VAL_EVERY = E('SPA_VAL_EVERY', 10.0)  # 분 — devval 평가 + best.pt
NVAL      = E('SPA_NVAL', 300)        # devval 평가 장수
SAVE_EVERY= E('SPA_SAVE_EVERY', 5.0)  # 분 — Drive 에 last.pt
LIMIT     = E('SPA_LIMIT', 0)         # >0 이면 zip 마다 앞 N장만(시험용)
EVAL_DEVVAL = E('SPA_EVAL_DEVVAL', 300)  # 7) devval 검출 채점 장수(0 = 생략)
THREADS   = E('SPA_THREADS', 2)
ENV = dict(os.environ, AIHUB_ROOT=AI, AIHUB_SPA_IMG=IMG, SPA_SEG_CACHE=CACHE, PYTHONPATH=CODE, PYTHONIOENCODING='utf-8')
os.environ.update(AIHUB_ROOT=AI, AIHUB_SPA_IMG=IMG, SPA_SEG_CACHE=CACHE)


def run(args, log=None, show=None):
    """python -m <args> 를 CODE 에서 실행, 줄마다 출력(show(line) 가 참인 줄만 화면에, log 파일엔 전부)."""
    p = subprocess.Popen([sys.executable, '-u', '-m'] + [str(a) for a in args], cwd=CODE, env=ENV,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    f = open(log, 'a', encoding='utf-8') if log else None
    for line in p.stdout:
        if f: f.write(line); f.flush()
        if show is None or show(line): print(line, end='', flush=True)
    rc = p.wait()
    if f: f.close()
    if rc: raise RuntimeError(f'실패(rc={rc}): {args}')


gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
print(gpu.stdout if gpu and gpu.stdout else 'GPU 없음 → 런타임 유형을 T4 GPU로 바꾸세요 (로컬 시험이면 무시)')
print('BASE', BASE, os.listdir(BASE))
print('결과 폴더', OUTD)

In [ ]:
# 2) 코드 복사 + 패키지
if os.path.exists(CODE): shutil.rmtree(CODE)
if os.path.exists(BASE + '/fire3dbeam/fire3dbeam'):                      # Drive 의 저장소 폴더
    shutil.copytree(BASE + '/fire3dbeam', CODE, ignore=shutil.ignore_patterns('.git', 'node_modules', 'out', 'out_*', '__pycache__'))
else:
    zipfile.ZipFile(BASE + '/FIRE-3D-BEAM-dev_github.zip').extractall(WORK)
need = ['evalr/spa_train.py', 'evalr/spa_colab.py', 'evalr/spa_held198.json', 'fire3dbeam/spa_seg.py', 'models/spa_seg.onnx', 'tools/aihub/build_index.py']
miss = [n for n in need if not os.path.exists(os.path.join(CODE, n))]
assert not miss, f'Drive 의 fire3dbeam/ 이 옛 버전입니다 — 새 파일을 올리세요: {miss}'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'onnx', 'onnxruntime', 'opencv-python-headless', 'shapely'], check=True)
sys.path.insert(0, CODE)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 3) 라벨 풀기 + 색인 (TL_SPA·VL_SPA, 1~3분)
os.makedirs(AI + '/labels', exist_ok=True)
for z in [f'{DATA}/Training/02.라벨링데이터/TL_SPA.zip', f'{DATA}/Validation/02.라벨링데이터/VL_SPA.zip']:
    d = AI + '/labels/' + os.path.basename(z)[:-4]
    if not os.path.exists(z): print('없음(건너뜀):', z); continue
    if os.path.exists(d + '/.done'): print('이미 풀림', d); continue
    zipfile.ZipFile(z).extractall(d); open(d + '/.done', 'w').close(); print('풀림', d)
run(['tools.aihub.build_index', AI])
idx = json.load(open(AI + '/label_index.json'))
from collections import Counter
print('SPA 라벨', Counter(v.split('/')[1] for k, v in idx.items() if '_SPA_' in k))

In [ ]:
# 4) TS_SPA 원천 → 학습 캐시(npz, 1/2 축척) + 평가용 2048 축소본(보류 198장·devval)  — 처음 약 25~40분
#    zip 마다 결과를 Drive 의 cache/<zip>.tar 로 보관 → 다시 실행하면 tar 에서 바로 복원(수십 초)
from evalr import spa_colab as SC, spa_data as SD
parts = sorted(glob.glob(f'{DATA}/Training/01.원천데이터/TS_SPA*.zip')) + sorted(glob.glob(f'{DATA}/Validation/01.원천데이터/VS_SPA*.zip'))
import zipfile
parts = [p for p in parts if os.path.getsize(p) > 0 and zipfile.is_zipfile(p)]   # 0바이트·받다 만 zip 제외(VS_SPA.zip 자리표시 등)
print('원천 zip', [os.path.basename(p) for p in parts])
for zp in parts:
    part = os.path.basename(zp)[:-4]
    tar = f'{OUTD}/cache/{part}{"_lim%d" % LIMIT if LIMIT else ""}.tar'
    done = f'{CACHE}/.{part}{"_lim%d" % LIMIT if LIMIT else ""}.done'
    if os.path.exists(done): print(part, '이미 처리됨'); continue
    if os.path.exists(tar):
        t0 = time.time(); n = SC.unpack(tar, IMG, CACHE); print(part, 'Drive 캐시에서 복원', n, f'{time.time() - t0:.0f}s')
    else:
        st = SC.extract_zip(zp, AI, IMG, CACHE, threads=3, limit=LIMIT or None)
        print(part, {k: v for k, v in st.items() if k not in ('files', 'meta')})
        if st.get('skipped'): continue
        try:
            sz = SC.pack(st, tar, IMG, CACHE); print(part, 'Drive 캐시 저장', tar, f'{sz / 1e6:.0f} MB')
        except Exception as e:
            print('Drive 캐시 저장 실패(용량?) — 학습은 계속 가능, 끊기면 다시 추출:', e)
    open(done, 'w').close()
S = SC.summary(IMG, CACHE); print(S)
assert S['held198_in_train'] == 0
if S['held198_png'] < 198: print(f"※ 보류 198장 중 {S['held198_png']}장만 영상 있음(원천 zip 에 없는 것은 평가에서 빠짐)")

## 5) 학습 (GPU, 시간 예산 기본 120분)
- `--width 2.0`(≈2.15M 파라미터, 기존 0.54M 의 4배), 배치 32, AMP(fp16), 384 자르기, 화질 흔들기(`--aug 1`), 10분마다 devval 300장 IoU → `best.pt`, 5분마다 `last.pt`(Drive).
- 연결이 끊기면 1)~4) 다시 실행 후 이 셀을 그대로 실행 — `last.pt` 가 있으면 자동으로 이어 학습(이미 끝났으면 바로 종료).
- 처음부터 다시 하려면 Drive 의 `colab_results/spa_gpu/ckpt/` 를 비우세요.
- 출력의 `data_wait` = 자료(CPU) 기다린 시간 비율. 50% 이상이면 CPU 병목이라 `WIDTH` 를 2.5~3.0 으로 올려도 스텝 수가 거의 줄지 않음(추론은 느려짐: 2.0 ≈ 기존의 1.6배).
- 예상: T4 에서 자료 병목 ≈ 초당 6~8 스텝(배치 32) → 120분 ≈ 4~5만 스텝(자르기 130만~180만 개, v2 CPU 60분의 약 40배).

In [ ]:
# 5) 학습
CK = OUTD + '/ckpt'
args = ['evalr.spa_train', 'train', '--device', DEVICE, '--width', WIDTH, '--bs', BS, '--lr', LR, '--workers', WORKERS,
        '--minutes', MINUTES, '--aug', 1, '--warmup', 200, '--per_img', 4, '--val_every', VAL_EVERY, '--nval', NVAL,
        '--save_every', SAVE_EVERY, '--log_every', E('SPA_LOG_EVERY', 500), '--threads', THREADS, '--out', CK]
if os.path.exists(CK + '/last.pt'): args += ['--resume', CK + '/last.pt']; print('이어 학습:', CK + '/last.pt')
t0 = time.time()
run(args, log=OUTD + '/train_log.txt')
print(f'학습 셀 {(time.time() - t0) / 60:.1f}분')

In [ ]:
# 6) ONNX 내보내기(best.pt = devval IoU 최고) → Drive
ONNX_NEW = OUTD + '/spa_seg_gpu.onnx'
run(['evalr.spa_train', 'export', '--ckpt', OUTD + '/ckpt/best.pt', '--onnx', ONNX_NEW,
     '--version', f'GPU(Colab) width {WIDTH} — TS_SPA 전체 개발 split(devval·보류 제외)'])
m = json.load(open(ONNX_NEW[:-5] + '.json', encoding='utf-8'))
print({k: m[k] for k in ('params', 'c', 'step', 'train_minutes', 'ntrain', 'size_mb', 'onnx_vs_torch_maxabs')})
print('devval', m['dev_val'])
# 추론 시간(CPU 1스레드, 정보카드 PC 기준 근사)
import cv2, numpy as np
from fire3dbeam import spa_seg as SS
SS._S.clear()                                     # 같은 경로로 다시 내보낸 경우 옛 세션 버림
its = SD.list_items('held198')[:3]
for pth in [CODE + '/models/spa_seg.onnx', ONNX_NEW]:
    t = []
    for it in its:
        g = cv2.imread(it['path'], 0); t0 = time.time(); r = SS.detect(g, path=pth); t.append(time.time() - t0)
    print(os.path.basename(pth), f'{np.mean(t[1:] or t):.2f} s/장', '실', len(r['rooms']), '계단', len(r['stairs']), '승강기', len(r['elevators']))

In [ ]:
# 7) 보류 198장 채점(이전 v2 보고와 같은 세트, 1회) + devval 검출 채점 — 기존 모델(v2) vs 새 모델
EV = WORK + '/spa_eval'
shutil.rmtree(EV + '/gpu_held198', ignore_errors=True); shutil.rmtree(EV + '/gpu_devval', ignore_errors=True)
for tag, onnx in [('v2_held198', CODE + '/models/spa_seg.onnx'), ('gpu_held198', ONNX_NEW)]:
    t0 = time.time(); run(['evalr.spa_eval', 'model', '--split', 'held198', '--tag', tag, '--onnx', onnx, '--cdir', EV]); print(tag, f'{time.time() - t0:.0f}s')
run(['evalr.spa_eval', 'score', '--split', 'held198', '--tag', 'v2_held198', '--tag2', 'gpu_held198', '--cdir', EV, '--json', OUTD + '/score_held198.json'])
if EVAL_DEVVAL:
    for tag, onnx in [('v2_devval', CODE + '/models/spa_seg.onnx'), ('gpu_devval', ONNX_NEW)]:
        run(['evalr.spa_eval', 'model', '--split', 'devval', '--tag', tag, '--onnx', onnx, '--cdir', EV, '--limit', EVAL_DEVVAL])
    run(['evalr.spa_eval', 'score', '--split', 'devval', '--tag', 'v2_devval', '--tag2', 'gpu_devval', '--cdir', EV, '--json', OUTD + '/score_devval.json'],
        show=lambda l: '###' in l or '소방 6종' in l or '정밀도' in l or '재현율' in l)
S = json.load(open(OUTD + '/score_held198.json', encoding='utf-8'))
print('\n보류 198장 요약 (로컬 v2 기준값: 소방6종 .748 · 실정밀도 .883 · 계단 R.801/P_lab.925 · 승강기 R.814/P_lab.786)')
for k, v in S.items():
    print(f"{k:12s} 영상 {v['n_img']} 소방6종 R {v['fire6_recall']:.3f} 실정밀도 {v['room_precision']:.3f} "
          f"계단 R {v['stairs']['recall']:.3f} P_lab {v['stairs']['precision_lab']:.3f} 승강기 R {v['elev']['recall']:.3f} P_lab {v['elev']['precision_lab']:.3f}")
print('결과:', sorted(os.listdir(OUTD)))

## 다음(로컬)
1. Drive `colab_results/spa_gpu/spa_seg_gpu.onnx`·`.json` 을 받아 `models/` 에 넣고 비교: 기존 v2 는 `models/spa_seg_v2.onnx`(+json) 로 이름을 바꿔 보관, 새 모델을 `models/spa_seg.onnx`(+`.json`) 로.
2. 계단 `min_score` 등 후처리 임계는 **devval 에서만** 다시 고르고(보류 금지), 고른 값은 새 `.json` 에 `"param": {"min_score": {"계단실": …}}` 로 넣으면 `fire3dbeam/spa_seg.detect` 가 모델별로 적용.
3. 보류 198장 수치(`score_held198.json`)는 1회만 보고 README 에 기록.